# S3.14 — show, printSchema, dtypes, count, describe, summary
**Date completed:** October 2026  
**Status:** Completed  
**Interview covered:** Q12 — show() vs collect() vs take()

In [0]:
# ============================================================
# S3.14 — Cell 2
# DataFrame Inspection, Actions, and Schema Introspection
#
# Goal:
# 1. Inspect schema using printSchema(), schema, dtypes, columns.
# 2. Use show() for limited visual inspection.
# 3. Compare first(), head(), take(), and collect().
# 4. Understand Driver-memory implications.
# 5. Use count() for total row count.
# 6. Use describe() and summary() for statistics.
# 7. Use explain() to inspect the execution plan.
# 8. Understand deterministic vs non-deterministic row retrieval.
# 9. Distinguish metadata inspection, transformations, and actions.
#
# Dataset:
# samples.tpch.orders
#
# Environment:
# Databricks Serverless
#
# Interview focus:
# show() vs collect() vs take() vs head() vs first()
# ============================================================

import pyspark.sql.functions as F


print("=== S3.14: DATAFRAME INSPECTION ===")
print()


# ------------------------------------------------------------
# Step 1 — Load Source DataFrame
# ------------------------------------------------------------

df_orders = spark.table("samples.tpch.orders")

print("Orders DataFrame loaded.")
print()


# ------------------------------------------------------------
# Step 2 — Schema and Metadata Inspection
#
# These operations inspect DataFrame metadata.
#
# They do not behave like full-data actions such as:
#
# count()
# collect()
#
# Useful properties/methods:
#
# printSchema()
# schema
# dtypes
# columns
# ------------------------------------------------------------

print("=== STEP 2: SCHEMA AND METADATA ===")


# ------------------------------------------------------------
# printSchema()
#
# Human-readable tree representation.
# ------------------------------------------------------------

print("\n--- printSchema() ---")

df_orders.printSchema()


# ------------------------------------------------------------
# columns
#
# Returns column names as a Python list.
# ------------------------------------------------------------

print("\n--- columns ---")

print(f"Type: {type(df_orders.columns)}")

print(f"Number of columns: {len(df_orders.columns)}")

print(df_orders.columns)

# ------------------------------------------------------------
# dtypes
#
# Returns:
#
# List[Tuple[column_name, datatype_string]]
# ------------------------------------------------------------

print("\n--- dtypes ---")

print(f"Type: {type(df_orders.dtypes)}")

for column_name, data_type in df_orders.dtypes:

    print(f"{column_name:<20} → {data_type}")


# ------------------------------------------------------------
# schema
#
# Returns a StructType object.
#
# Useful when schema information needs to be accessed
# programmatically.
# ------------------------------------------------------------

print("\n--- schema ---")

print(f"Schema object type: {type(df_orders.schema)}")

print(df_orders.schema)

print("\nIndividual schema fields:")

for field in df_orders.schema.fields:

    print(
        f"{field.name:<20} "
        f"type={field.dataType.simpleString():<15} "
        f"nullable={field.nullable}"
    )

print()

# ------------------------------------------------------------
# Step 3 — show(): Limited Visual Inspection
#
# show() displays rows in notebook/console output.
#
# It is an ACTION because Spark must produce records.
#
# Important:
#
# show(n) limits the number of rows displayed.
#
# However, do not assume the underlying computation is always
# cheap. Spark may still need substantial work depending on
# the transformations before show().
#
# show() prints the result and returns None.
# ------------------------------------------------------------

print("=== STEP 3: show() ===")

df_preview = df_orders.select(
    "o_orderkey",
    "o_orderstatus",
    "o_totalprice",
    "o_orderdate"
)


# Default:
#
# show()
# → 20 rows by default

print("\n--- Default show() ---")

show_result = df_preview.show()

print(f"Return value from show(): {show_result}")

# ------------------------------------------------------------
# Custom number of rows
# ------------------------------------------------------------

print("\n--- show(5, truncate=False) ---")

df_preview.show(5, truncate=False)


# ------------------------------------------------------------
# Vertical output
#
# Useful when a DataFrame has many or wide columns.
# ------------------------------------------------------------

print("\n--- show(3, vertical=True) ---")

df_preview.show(3, vertical=True)

print()


In [0]:
# ------------------------------------------------------------
# Step 4 — first()
#
# first() returns one Row object.
#
# Equivalent conceptually to retrieving one record.
#
# Important:
# Without orderBy(), do not interpret it as the
# business-defined "first" record.
# ------------------------------------------------------------

print("=== STEP 4: first() ===")


first_row = df_preview.first()


print(f"Return type: {type(first_row)}")

print(f"Row: {first_row}")

print(f"Order key: {first_row['o_orderkey']}")

print()

In [0]:
# ------------------------------------------------------------
# Step 5 — head()
#
# head() without a number:
#
# → returns one Row
#
# head(n):
#
# → returns a Python list of Row objects
#
# Like first()/take(), ordering is not business-deterministic
# unless an explicit orderBy() is applied.
# ------------------------------------------------------------

print("=== STEP 5: head() ===")


head_one = df_preview.head()

print(f"head() type: {type(head_one)}")

print(f"head() result: {head_one}")


head_three = df_preview.head(3)

print(f"\nhead(3) type: {type(head_three)}")

print(f"head(3) length: {len(head_three)}")

print(f"head(3) result: {head_three}")

print()


In [0]:
# ------------------------------------------------------------
# Step 6 — take()
#
# take(n):
#
# Returns up to n rows to Python as:
#
# List[Row]
#
# Useful when the application actually needs a small number
# of Row objects rather than only displaying them.
# ------------------------------------------------------------

print("=== STEP 6: take() ===")


first_5 = df_preview.take(
    5
)

print(
    f"take(5) type: {type(first_5)}"
)

print(
    f"Number of rows returned: {len(first_5)}"
)

print(f"First Row type: {type(first_5[0])}")

print(
    f"First Row: {first_5[0]}"
)

print(
    f"Order key: {first_5[0]['o_orderkey']}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 7 — collect()
#
# collect():
#
# Returns ALL rows from the DataFrame result to the Driver
# as a Python list of Row objects.
#
# This is safe only when the result is known to be small.
#
# We deliberately use collect() on a small result.
# ------------------------------------------------------------

print("=== STEP 7: collect() ===")

df_statuses = (
    df_orders

    .select(
        "o_orderstatus"
    ).distinct())

status_count = df_statuses.count()

print(f"Distinct status count: {status_count}")

status_list = df_statuses.collect()

print(f"collect() result type: {type(status_list)}")

print(f"Number of collected Rows: {len(status_list)}")

print(f"Collected values: {status_list}")

print()

In [0]:
# ------------------------------------------------------------
# Step 8 — collect() vs take()
#
# collect():
# → ALL rows in result.
#
# take(n):
# → Up to n rows.
#
# Both return Row objects to Python/Driver.
# ------------------------------------------------------------

print(
    "=== STEP 8: collect() vs take() ==="
)


print("""
collect():

df.collect()

→ Returns ALL result rows.
→ Python type: list[Row]
→ Driver memory depends on the full result size.


take(n):

df.take(n)

→ Returns up to n rows.
→ Python type: list[Row]
→ Useful when only a small number of records is required.


Key rule:

collect()
→ result size must already be known to be small.

take(n)
→ explicitly limits returned row count.
""")

In [0]:
# ------------------------------------------------------------
# Step 9 — Deterministic "First" Rows
#
# first(), head(), take(), show(), and limit()
# do NOT inherently define business ordering.
#
# If the requirement is:
#
# "Get the five highest-value orders"
#
# define the ordering first.
# ------------------------------------------------------------

print(
    "=== STEP 9: ORDERED ROW RETRIEVAL ==="
)

top_5_orders = (

    df_orders

    .select(
        "o_orderkey",
        "o_totalprice"
    )

    .orderBy(
        F.col(
            "o_totalprice"
        ).desc()
    )

    .take(
        5
    )
)

print("Five highest-value orders:")

for row in top_5_orders:

    print(
        row)

print()

In [0]:
# ------------------------------------------------------------
# Step 10 — count()
#
# count() returns the total number of rows as an integer.
#
# It is an ACTION.
#
# Spark must process enough of the DataFrame plan to determine
# the total record count.
# ------------------------------------------------------------

print("=== STEP 10: count() ===")

total_orders = df_orders.count()

print(f"Total orders: {total_orders:,}")

print(f"count() return type: {type(total_orders)}")

print()

In [0]:
# ------------------------------------------------------------
# Step 11 — describe()
#
# describe() produces a DataFrame containing basic statistics.
#
# Typically includes:
#
# count
# mean
# stddev
# min
# max
#
# describe() itself returns a DataFrame.
#
# show() is what requires the statistics result to be produced.
# ------------------------------------------------------------

print("=== STEP 11: describe() ===")

df_description = (
    df_orders

    .select(
        "o_totalprice",
        "o_shippriority"
    ).describe())

print(
    f"describe() return type: "
    f"{type(df_description)}"
)

df_description.show(
    truncate=False
)

print()

In [0]:
# ------------------------------------------------------------
# Step 12 — summary()
#
# summary() provides additional statistical summaries.
#
# Default statistics commonly include:
#
# count
# mean
# stddev
# min
# 25%
# 50%
# 75%
# max
#
# summary() also returns a DataFrame.
# ------------------------------------------------------------

print("=== STEP 12: summary() ===")

df_summary = (
    df_orders

    .select(
        "o_totalprice"
    ).summary())

print(f"summary() return type: "f"{type(df_summary)}")


df_summary.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 13 — Custom summary()
#
# We can request only specific statistics.
# ------------------------------------------------------------

print(
    "=== STEP 13: CUSTOM summary() ==="
)

(
    df_orders.select("o_totalprice")

    .summary(
        "count",
        "min",
        "25%",
        "50%",
        "75%",
        "max"
    ).show(truncate=False))

print()

In [0]:
# ------------------------------------------------------------
# Step 14 — explain()
#
# explain() shows Spark's query/execution plan.
#
# Useful for understanding:
#
# - projection
# - filters
# - aggregation
# - exchanges/shuffles
# - physical execution strategy
#
# This is an inspection/debugging tool.
# ------------------------------------------------------------

print("=== STEP 14: explain() ===")

df_plan_example = (

    df_orders
    .select("o_orderstatus", "o_totalprice")

    .filter(F.col("o_totalprice") > 100000)

    .groupBy("o_orderstatus")

    .agg(F.avg("o_totalprice").alias("avg_price")))

print("\nQuery plan:")

df_plan_example.explain()

print()

In [0]:
# ------------------------------------------------------------
# Step 15 — Driver-Memory Risk
#
# DO NOT execute:
#
# df_orders.collect()
#
# on a large DataFrame simply to inspect it.
#
# collect() materialises the entire result on the Driver.
#
# The result can exceed Driver memory.
# ------------------------------------------------------------

print(
    "=== STEP 15: collect() DRIVER-MEMORY RISK ==="
)


print(
    f"Current orders row count: {total_orders:,}"
)


print("""
We intentionally DO NOT run:

df_orders.collect()

Why?

collect()
→ returns every row to the Driver.

If the resulting Python objects exceed available Driver memory,
the application may become extremely slow or fail with memory-related
errors.

The danger depends on:

- number of rows
- number of columns
- data types
- size of strings / nested values
- Driver memory
""")


# ------------------------------------------------------------
# Safer inspection patterns
# ------------------------------------------------------------

print("Safer alternatives for inspection:")

print("""
df.show(10)

→ Display a limited number of rows.

df.take(10)

→ Return a limited number of Rows to Python.


df.limit(100).collect()

→ Collect a deliberately limited result.


df.select(...).distinct().collect()

→ Appropriate only when you KNOW the distinct result is small.
""")

In [0]:
# ------------------------------------------------------------
# Step 16 — Important: limit().collect()
#
# limit() is a transformation.
#
# collect() is the action.
#
# Therefore:
#
# df.limit(100).collect()
#
# means:
#
# Restrict result
#       ↓
# Collect the limited result to Driver
# ------------------------------------------------------------

print("=== STEP 16: LIMITED collect() ===")

limited_rows = (
    df_orders

    .select(
        "o_orderkey",
        "o_orderstatus"
    ).limit(5).collect())

print(f"Limited collect row count: {len(limited_rows)}")

print(limited_rows)

print()

In [0]:
# ------------------------------------------------------------
# Step 17 — Result-Type Comparison
# ------------------------------------------------------------

print(
    "=== STEP 17: RESULT-TYPE COMPARISON ==="
)


print("""
printSchema()
→ Prints schema tree.

columns
→ Python list of column names.

dtypes
→ Python list of (name, type) tuples.

schema
→ StructType.

show()
→ Displays rows.
→ Returns None.

first()
→ Returns one Row.

head()
→ Returns one Row.

head(n)
→ Returns list[Row].

take(n)
→ Returns list[Row].

collect()
→ Returns ALL result rows as list[Row].

count()
→ Returns Python integer.

describe()
→ Returns DataFrame.

summary()
→ Returns DataFrame.

explain()
→ Prints query/execution plan.
""")

In [0]:
# ------------------------------------------------------------
# Step 18 — Transformations, Actions, and Metadata
# ------------------------------------------------------------

print(
    "=== STEP 18: CLASSIFICATION ==="
)


print("""
DATAFRAME TRANSFORMATIONS:

select()
filter()
groupBy() / agg()
orderBy()
limit()
distinct()

These build or extend a DataFrame execution plan.


ACTIONS / RESULT RETRIEVAL:

show()
collect()
take()
first()
head()
count()

These require Spark to produce data/results.


METADATA / PLAN INSPECTION:

printSchema()
schema
dtypes
columns
explain()

These are used to inspect schema or plan information.


STATISTICAL DATAFRAMES:

describe()
summary()

These return DataFrames.

Their results are computed when an action such as show()
is called.
""")

In [0]:
# ------------------------------------------------------------
# Step 19 — Interview Comparison
# ------------------------------------------------------------

print(
    "=== STEP 19: INTERVIEW COMPARISON ==="
)


print("""
show()

Purpose:
→ Human-readable preview.

Result:
→ Printed output.

Driver exposure:
→ Limited displayed rows.

Use:
→ Notebook/debug inspection.


collect()

Purpose:
→ Retrieve every result row into Python.

Result:
→ list[Row].

Driver exposure:
→ Entire result.

Use:
→ Only when result is known to be small.


take(n)

Purpose:
→ Retrieve up to n Rows.

Result:
→ list[Row].

Driver exposure:
→ Limited to n rows.

Use:
→ Programmatic small-row retrieval.


first()

Purpose:
→ Retrieve one Row.

Result:
→ Row.

head()

Purpose:
→ Retrieve one row or n rows.

Result:

head()
→ Row

head(n)
→ list[Row]


Important:

Without orderBy(), none of these should be interpreted as
business-defined first rows.
""")

In [0]:
# ------------------------------------------------------------
# Step 20 — Final Learning Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
Schema inspection:

printSchema()
→ Human-readable schema.

schema
→ StructType representation.

dtypes
→ Column/type tuples.

columns
→ Column-name list.


Row inspection:

show()
→ Display small result.

first()
→ One Row.

head()
→ One Row.

head(n)
→ Up to n Rows.

take(n)
→ Up to n Rows.

collect()
→ ALL result rows.


Counting:

count()
→ Total row count.


Statistics:

describe()
→ Basic descriptive statistics.

summary()
→ Extended/configurable summary statistics.


Execution inspection:

explain()
→ Inspect Spark query/execution plan.


Critical Driver rule:

Never use collect() on a large DataFrame unless the result size
is known to fit safely in Driver memory.


Ordering rule:

show()
first()
head()
take()
limit()

do NOT define business ordering.

For deterministic Top-N logic:

orderBy(...)
    ↓
take(n)

or:

orderBy(...)
    ↓
limit(n)


Core inspection principle:

Need to SEE rows?
→ show()

Need Rows in Python?
→ take(n)

Need ONE Row?
→ first()

Need ALL rows?
→ collect() only when result is known to be small

Need total number of rows?
→ count()

Need schema?
→ printSchema() / schema / dtypes / columns

Need statistics?
→ describe() / summary()

Need execution plan?
→ explain()
""")


print("=== S3.14 CELL 2 COMPLETED ===")